In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats

from ML_xgboost import train_xgboost_model
from ML_lightgbm import train_lightgbm_model
from ML_catboost import train_catboost_model
from ML_randomforest import train_randomforest_model

TEST_SIZE = 0.20
VAL_SIZE = 0.20
SEED_LIST = [42, 123, 256, 512, 777, 1024, 2024, 2025,
             3407, 6666, 8888, 10086, 16384, 20516, 27182, 31415]
OUTPUT_DIR = "D:/1.SJTU/MentalHealthProjects/ML_CLPN_Project/03_Output/model_stability_anxiety"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Load data
df_model = pd.read_csv("D:/1.SJTU/MentalHealthProjects/ML_CLPN_Project/01_Input/df_model.csv")

selected_vars = [
    "somatic_y1", "BMI_T1_cat", "sleep_dura_T1_cat", "sleep_quali_T1", "insomnia_y1",
    "life_satis_y1", "ms_ses_y1", "per_stress_y1", "ms_stress_y1", "depression_y1", "anxiety_y1",
    "ace", "loneliness_y1", "support_y1", "gender_T1", "age_T1", "residence", "income",
    "income_ineqCity_y1", "sss_now", "marrige_par_bin", "edu_pa",
    "eat_unctl_y1", "eat_emot_y1", "food_sweetdrink_T1", "food_takeout_T1",
    "IPAQ_T1_1_bin", "IPAQ_T1_3_bin", "IPAQ_T1_5_bin", "screenT_weekday_T1", "screenT_weekend_T1",
    "psmu_y1", "media_BadMood_T1", "media_GoodMood_T1", "edu_self", "grade_T1"
]

y_col = "anxiety_y2"
y_cont = pd.to_numeric(df_model[y_col], errors="coerce")
y = (y_cont >= 5).astype(int)
X = df_model[selected_vars].copy()

print(f"Sample size: {len(X)}")
print(f"Positive class rate: {y.mean():.4f}")


# 2. Model registry
model_trainers = {
    "XGBoost": train_xgboost_model,
    "LightGBM": train_lightgbm_model,
    "CatBoost": train_catboost_model,
    "RandomForest": train_randomforest_model
}

# 3. Functions
def calc_ci(series, confidence=0.95):
    """
    Calculate mean, SD, SE, and confidence interval.
    """
    series = pd.Series(series).dropna()
    n = len(series)
    mean_val = series.mean()
    sd_val = series.std(ddof=1) if n > 1 else 0.0
    se_val = sd_val / np.sqrt(n) if n > 1 else 0.0

    if n > 1:
        t_value = stats.t.ppf((1 + confidence) / 2, df=n - 1)
        ci_low = mean_val - t_value * se_val
        ci_high = mean_val + t_value * se_val
    else:
        ci_low, ci_high = mean_val, mean_val

    return {
        "n_runs": n,
        "mean": mean_val,
        "sd": sd_val,
        "se": se_val,
        "ci_low": ci_low,
        "ci_high": ci_high
    }


def summarize_metric(df, group_cols, metric_col):
    """
    Summarize one metric by group.
    """
    summary_rows = []
    grouped = df.groupby(group_cols)

    for group_name, group_df in grouped:
        stats_dict = calc_ci(group_df[metric_col])

        if not isinstance(group_name, tuple):
            group_name = (group_name,)

        row = dict(zip(group_cols, group_name))
        row.update({
            "Metric": metric_col,
            "Mean": stats_dict["mean"],
            "SD": stats_dict["sd"],
            "SE": stats_dict["se"],
            "CI95_Low": stats_dict["ci_low"],
            "CI95_High": stats_dict["ci_high"],
            "N_runs": stats_dict["n_runs"],
            "Mean_SD": f"{stats_dict['mean']:.3f} ± {stats_dict['sd']:.3f}",
            "Mean_CI95": f"{stats_dict['mean']:.3f} ({stats_dict['ci_low']:.3f}, {stats_dict['ci_high']:.3f})"
        })
        summary_rows.append(row)

    return pd.DataFrame(summary_rows)


# 4. Multi-seed training
all_results = []

for model_name, trainer in model_trainers.items():
    print("=" * 60)
    print(f"Running model: {model_name}")

    for seed in SEED_LIST:
        print(f"  Seed = {seed}")

        try:
            res = trainer(
                X=X,
                y=y,
                selected_vars=selected_vars,
                random_state=seed,
                test_size=TEST_SIZE,
                val_size=VAL_SIZE
            )

            row_train = {
                "Model": model_name,
                "Seed": seed,
                "Dataset": "Train",
                "AUC": res.get("train_auc", np.nan),
                "F1": res.get("train_f1", np.nan),
                "Accuracy": res.get("train_acc", np.nan)
            }

            row_val = {
                "Model": model_name,
                "Seed": seed,
                "Dataset": "Validation",
                "AUC": res.get("val_auc", np.nan),
                "F1": res.get("val_f1", np.nan),
                "Accuracy": res.get("val_acc", np.nan)
            }

            row_test = {
                "Model": model_name,
                "Seed": seed,
                "Dataset": "Test",
                "AUC": res.get("test_auc", np.nan),
                "F1": res.get("test_f1", np.nan),
                "Accuracy": res.get("test_acc", np.nan)
            }

            all_results.extend([row_train, row_val, row_test])

        except Exception as e:
            print(f"  Error under seed {seed} for {model_name}: {e}")

# 5. Save raw results
df_all_results = pd.DataFrame(all_results)
raw_path = os.path.join(OUTPUT_DIR, "multi_seed_raw_results.csv")
df_all_results.to_csv(raw_path, index=False, encoding="utf-8-sig")

print("\nRaw results saved to:")
print(raw_path)

# 6. Summary results
summary_auc = summarize_metric(df_all_results, ["Model", "Dataset"], "AUC")
summary_f1 = summarize_metric(df_all_results, ["Model", "Dataset"], "F1")
summary_acc = summarize_metric(df_all_results, ["Model", "Dataset"], "Accuracy")

df_summary = pd.concat([summary_auc, summary_f1, summary_acc], axis=0, ignore_index=True)

summary_path = os.path.join(OUTPUT_DIR, "multi_seed_summary_results.csv")
df_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nSummary results saved to:")
print(summary_path)

# 7. Make a wide table
df_summary_for_paper = df_summary.pivot_table(
    index=["Model", "Dataset"],
    columns="Metric",
    values="Mean_SD",
    aggfunc="first"
).reset_index()

paper_table_path = os.path.join(OUTPUT_DIR, "multi_seed_summary_for_paper.csv")
df_summary_for_paper.to_csv(paper_table_path, index=False, encoding="utf-8-sig")

print("\nPaper-style summary table saved to:")
print(paper_table_path)

# 8. Identify the most stable best model on test AUC
test_auc_summary = summary_auc[summary_auc["Dataset"] == "Test"].copy()
test_auc_summary = test_auc_summary.sort_values(by=["Mean", "SD"], ascending=[False, True])

print("\n=== Test AUC summary (sorted by mean desc, SD asc) ===")
print(test_auc_summary[["Model", "Mean", "SD", "CI95_Low", "CI95_High", "Mean_SD", "Mean_CI95"]])

Sample size: 1077
Positive class rate: 0.6072
Running model: XGBoost
  Seed = 42
Start hyperparameter search...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

=== RandomizedSearchCV ===
Best CV AUC: 0.770076652934423
Best Params: {'xgb__subsample': 0.5, 'xgb__scale_pos_weight': 1.2, 'xgb__reg_lambda': 3, 'xgb__reg_alpha': 7, 'xgb__n_estimators': 400, 'xgb__min_child_weight': 6, 'xgb__max_depth': 4, 'xgb__learning_rate': 0.01, 'xgb__gamma': 0.3, 'xgb__colsample_bytree': 0.7, 'xgb__colsample_bylevel': 0.5}
[0]	train-logloss:0.67134	eval-logloss:0.67222
[50]	train-logloss:0.62184	eval-logloss:0.63373
[100]	train-logloss:0.58947	eval-logloss:0.61027
[150]	train-logloss:0.56961	eval-logloss:0.59711
[200]	train-logloss:0.55619	eval-logloss:0.58828


C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [08:55:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[250]	train-logloss:0.54567	eval-logloss:0.58375
[300]	train-logloss:0.53788	eval-logloss:0.58029
[350]	train-logloss:0.53315	eval-logloss:0.57814
[399]	train-logloss:0.52897	eval-logloss:0.57682
  Seed = 123
Start hyperparameter search...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

=== RandomizedSearchCV ===
Best CV AUC: 0.7850788156824154
Best Params: {'xgb__subsample': 0.5, 'xgb__scale_pos_weight': 0.8, 'xgb__reg_lambda': 10, 'xgb__reg_alpha': 5, 'xgb__n_estimators': 800, 'xgb__min_child_weight': 10, 'xgb__max_depth': 2, 'xgb__learning_rate': 0.05, 'xgb__gamma': 1.0, 'xgb__colsample_bytree': 0.5, 'xgb__colsample_bylevel': 0.4}
[0]	train-logloss:0.67087	eval-logloss:0.67105
[50]	train-logloss:0.57823	eval-logloss:0.56806
[100]	train-logloss:0.55498	eval-logloss:0.54917
[150]	train-logloss:0.55018	eval-logloss:0.54420
[200]	train-logloss:0.54451	eval-logloss:0.54033
[250]	train-logloss:0.54074	eval-logloss:0.53825
[300]	train-logloss:0.53884	eval-logloss:0.53569


C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [08:56:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[350]	train-logloss:0.53821	eval-logloss:0.53480
[400]	train-logloss:0.53752	eval-logloss:0.53370
[450]	train-logloss:0.53549	eval-logloss:0.53272
[500]	train-logloss:0.53464	eval-logloss:0.53309
[504]	train-logloss:0.53489	eval-logloss:0.53327
  Seed = 256
Start hyperparameter search...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

=== RandomizedSearchCV ===
Best CV AUC: 0.7603539509948642
Best Params: {'xgb__subsample': 0.5, 'xgb__scale_pos_weight': 1.2, 'xgb__reg_lambda': 3, 'xgb__reg_alpha': 5, 'xgb__n_estimators': 600, 'xgb__min_child_weight': 20, 'xgb__max_depth': 3, 'xgb__learning_rate': 0.08, 'xgb__gamma': 1.0, 'xgb__colsample_bytree': 0.4, 'xgb__colsample_bylevel': 0.4}
[0]	train-logloss:0.67080	eval-logloss:0.67144
[50]	train-logloss:0.56247	eval-logloss:0.57845
[100]	train-logloss:0.54843	eval-logloss:0.56548
[150]	train-logloss:0.54281	eval-logloss:0.56395
[158]	train-logloss:0.54271	eval-logloss:0.56388


...


=== RandomizedSearchCV Results ===
Best CV AUC: 0.7706
Best Params: {'lgbm__subsample_freq': 1, 'lgbm__subsample': 0.6, 'lgbm__scale_pos_weight': 2, 'lgbm__reg_lambda': 1, 'lgbm__reg_alpha': 0, 'lgbm__objective': 'binary', 'lgbm__num_leaves': 50, 'lgbm__n_estimators': 500, 'lgbm__min_child_samples': 45, 'lgbm__max_depth': -1, 'lgbm__learning_rate': 0.01, 'lgbm__colsample_bytree': 0.7, 'lgbm__boosting_type': 'dart'}

Training final model with Early Stopping (Patience=100)...
[200]	training's binary_logloss: 0.592817	valid_1's binary_logloss: 0.604753


C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


[400]	training's binary_logloss: 0.566909	valid_1's binary_logloss: 0.584802
[600]	training's binary_logloss: 0.556914	valid_1's binary_logloss: 0.578479
[800]	training's binary_logloss: 0.541375	valid_1's binary_logloss: 0.571399
[1000]	training's binary_logloss: 0.527245	valid_1's binary_logloss: 0.564883
[1200]	training's binary_logloss: 0.516588	valid_1's binary_logloss: 0.560379
[1400]	training's binary_logloss: 0.503805	valid_1's binary_logloss: 0.557747
[1600]	training's binary_logloss: 0.492736	valid_1's binary_logloss: 0.557269
[1800]	training's binary_logloss: 0.481623	valid_1's binary_logloss: 0.555504
[2000]	training's binary_logloss: 0.468992	valid_1's binary_logloss: 0.553341
[2200]	training's binary_logloss: 0.457559	valid_1's binary_logloss: 0.550353
[2400]	training's binary_logloss: 0.446251	valid_1's binary_logloss: 0.548955
[2600]	training's binary_logloss: 0.435215	valid_1's binary_logloss: 0.547284
[2800]	training's binary_logloss: 0.424253	valid_1's binary_logloss

C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== CatBoost RandomizedSearchCV ===
Best CV AUC: 0.7665
Best Params: {'cat__subsample': 0.7, 'cat__scale_pos_weight': 3, 'cat__rsm': 0.7, 'cat__learning_rate': 0.01, 'cat__l2_leaf_reg': 7, 'cat__iterations': 500, 'cat__depth': 4}

Training final CatBoost model with Early Stopping...
0:	test: 0.5484594	best: 0.5484594 (0)	total: 2.15ms	remaining: 4.3s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7642857143
bestIteration = 13

Shrink model to first 14 iterations.
  Seed = 123
Start CatBoost hyperparameter search...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

=== CatBoost RandomizedSearchCV ===
Best CV AUC: 0.7783
Best Params: {'cat__subsample': 0.7, 'cat__scale_pos_weight': 2, 'cat__rsm': 0.7, 'cat__learning_rate': 0.01, 'cat__l2_leaf_reg': 3, 'cat__iterations': 500, 'cat__depth': 4}

Training final CatBoost model with Early Stopping...
0:	test: 0.7486695	best: 0.7486695 (0)	total: 1.5ms	remaining: 2.99s
Stopped by overfitting detector  (50 iter

## 带误差条的性能比较图

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os

# === Load data from CSV ===
csv_path = r"D:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\model_stability_anxiety\multi_seed_summary_results.csv"
df_raw = pd.read_csv(csv_path)

# Normalize model names
df_raw['Model'] = df_raw['Model'].replace({
    'RandomFo': 'RF',
    'RandomForest': 'RF',
    'LightGBM': 'LGBM',
})

# Build long-form df for mean and SD
MODEL_ORDER = ['CatBoost', 'LGBM', 'RF', 'XGBoost']
DATASET_ORDER = ['Train', 'Validation', 'Test']

df_raw['Model'] = pd.Categorical(df_raw['Model'], categories=MODEL_ORDER, ordered=True)
df_raw['Dataset'] = pd.Categorical(df_raw['Dataset'], categories=DATASET_ORDER, ordered=True)
df_raw = df_raw.sort_values(['Metric', 'Model', 'Dataset'])

# === Plot settings (unchanged) ===
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'font.size': 18
})

fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=600, tight_layout=True)

METRICS = ['AUC', 'F1', 'Accuracy']
PALETTES = ['Greens', 'Oranges', 'Purples']
TITLES = ['AUC Performance Comparison', 'F1 Score Performance Comparison', 'Accuracy Performance Comparison']
YLABELS = ['AUC Score', 'F1 Score', 'Accuracy Score']
ALPHA = 0.75  # bar transparency

for ax, metric, palette_name, title, ylabel in zip(axes, METRICS, PALETTES, TITLES, YLABELS):
    sub = df_raw[df_raw['Metric'] == metric].copy()

    n_models = len(MODEL_ORDER)
    n_datasets = len(DATASET_ORDER)
    bar_width = 0.35          # width of each individual bar
    group_width = bar_width * n_datasets   # no gap: groups touch each other internally
    between_gap = 0.30        # gap between model groups

    # x center for each model group
    group_positions = np.arange(n_models) * (group_width + between_gap)
    # offsets within group: bars touch each other (no gap)
    offsets = np.array([-bar_width, 0, bar_width])

    # Get palette colors
    cmap = plt.get_cmap(palette_name)
    # Sample 3 evenly spaced colors from the colormap (avoid too light/too dark)
    colors = [cmap(v) for v in [0.35, 0.6, 0.85]]

    for di, (dataset, color) in enumerate(zip(DATASET_ORDER, colors)):
        means, sds = [], []
        for model in MODEL_ORDER:
            row = sub[(sub['Model'] == model) & (sub['Dataset'] == dataset)]
            means.append(float(row['Mean'].values[0]))
            sds.append(float(row['SD'].values[0]))

        x_pos = group_positions + offsets[di]

        # Draw bars with alpha (半透明)
        ax.bar(
            x_pos, means,
            width=bar_width,          # exactly bar_width, no gap
            color=color,
            alpha=ALPHA,
            label=dataset,
            zorder=3,
            linewidth=0              # no edge line so bars look seamless
        )

        # Error bars
        ax.errorbar(
            x_pos, means,
            yerr=sds,
            fmt='none',
            ecolor='black',
            elinewidth=1.2,
            capsize=4,
            capthick=1.2,
            zorder=4
        )

    ax.set_ylim(0, 1)
    margin = group_width * 0.6
    ax.set_xlim(group_positions[0] - margin, group_positions[-1] + margin)
    ax.set_xticks(group_positions)
    ax.set_xticklabels(MODEL_ORDER)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', length=0)     # remove x tick marks only
    ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)  # y grid only
    ax.grid(axis='x', visible=False)        # no x grid

    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7
    )

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

OUTPUT_DIR = r"D:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\model_stability_anxiety"
png_save_path = os.path.join(OUTPUT_DIR, "Performance_Comparison_WithErrorBars.png")
plt.savefig(png_save_path, dpi=600, bbox_inches='tight')
plt.close()
print(f"Saved to:\n{png_save_path}")

Saved to:
D:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\model_stability_anxiety\Performance_Comparison_WithErrorBars.png


In [2]:
# import pandas as pd
# import numpy as np
# from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
# import xgboost as xgb
# import matplotlib.pyplot as plt
# import seaborn as sns
# from ML_xgboost import train_xgboost_model
# from ML_lightgbm import train_lightgbm_model
# from ML_catboost import train_catboost_model
# from ML_randomforest import train_randomforest_model 

# df_model = pd.read_csv("D:/1.SJTU/MentalHealthProjects/ML_CLPN_Project/01_Input/df_model.csv")

# selected_vars = [
#     "somatic_y1", "BMI_T1_cat", "sleep_dura_T1_cat", "sleep_quali_T1", "insomnia_y1",
#     "life_satis_y1", "ms_ses_y1", "per_stress_y1", "ms_stress_y1", "depression_y1", "anxiety_y1",
#     "ace", "loneliness_y1", "support_y1", "gender_T1", "age_T1", "residence", "income", 
#      "income_ineqCity_y1", "sss_now", "marrige_par_bin", "edu_pa",
#     "eat_unctl_y1", "eat_emot_y1", "food_sweetdrink_T1", "food_takeout_T1",
#     "IPAQ_T1_1_bin", "IPAQ_T1_3_bin", "IPAQ_T1_5_bin", "screenT_weekday_T1", "screenT_weekend_T1",
#     "psmu_y1", "media_BadMood_T1", "media_GoodMood_T1", "edu_self","grade_T1"
# ]
# y_col = "anxiety_y2"
# y_cont = pd.to_numeric(df_model[y_col], errors="coerce")

# y = (y_cont >= 5).astype(int)
# X = df_model[selected_vars].copy()

# RANDOM_STATE = 6666
# TEST_SIZE = 0.20
# VAL_SIZE = 0.20

# # === 1. Training XGBoost ===
# print("=== 1. Training XGBoost ===")
# res_xgb = train_xgboost_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
# print(f"XGB Test AUC: {res_xgb['test_auc']:.4f}")

# # === 2. Training LightGBM ===
# print("\n=== 2. Training LightGBM ===")
# res_lgbm = train_lightgbm_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
# print(f"LGBM Test AUC: {res_lgbm['test_auc']:.4f}")

# # === 3. Training CatBoost ===
# print("\n=== 3. Training CatBoost ===")
# res_cat = train_catboost_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
# print(f"CatBoost Test AUC: {res_cat['test_auc']:.4f}")

# # === 4. Training Random Forest ===
# print("\n=== 4. Training RandomForest ===")
# res_rf = train_randomforest_model(X, y, selected_vars, RANDOM_STATE, TEST_SIZE, VAL_SIZE)
# print(f"RF Test AUC: {res_rf['test_auc']:.4f}")

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

models = ['RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']

# AUC Score: higher is better
auc_scores = [
    res_rf['test_auc'],
    res_xgb['test_auc'],
    res_lgbm['test_auc'],
    res_cat['test_auc']
]

# F1 Score: higher is better (threshold-dependent)
f1_scores = [
    res_rf['test_f1'],
    res_xgb['test_f1'],
    res_lgbm['test_f1'],
    res_cat['test_f1']
]

# ACC Score: higher is better (threshold-dependent)
acc_scores = [
    res_rf['test_acc'],
    res_xgb['test_acc'],
    res_lgbm['test_acc'],
    res_cat['test_acc']
]

# create DataFrame
df_perf = pd.DataFrame({
    'Model': models,
    'Test AUC': auc_scores,
    'Test F1': f1_scores,
    'Test Accuracy': acc_scores
})

plt.rcParams.update({
    'font.family': 'Times New Roman',
    'axes.titlesize': 22,
    'axes.labelsize': 20,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'font.size': 18
})

fig, axes = plt.subplots(1, 3, figsize=(22, 8), dpi=600, tight_layout=True)

# AUC plot
sns.barplot(x='Model', y='Test AUC', data=df_perf, ax=axes[0], palette="Greens")
axes[0].set_title(f'Test Set AUC Comparison')
axes[0].set_ylabel('AUC')
axes[0].tick_params(axis='x', labelsize=16) # rotation=15
axes[0].set_ylim(0.5, 1)
for i in axes[0].containers:
    axes[0].bar_label(i, fmt='%.4f', label_type='edge', padding=4, fontsize=14)
axes[0].margins(y=0.1)
axes[0].legend(
    loc='upper right',
    bbox_to_anchor=(1, 1),
    framealpha=0.7     # 半透明
)
# F1 plot
sns.barplot(x='Model', y='Test F1', data=df_perf, ax=axes[1], palette="Oranges")
axes[1].set_title(f'Test Set F1-score Comparison')
axes[1].set_ylabel('F1-score')
axes[1].tick_params(axis='x', labelsize=16)
axes[1].set_ylim(0.5, 1)
for i in axes[1].containers:
    axes[1].bar_label(i, fmt='%.4f', label_type='edge', padding=4, fontsize=14)
axes[1].margins(y=0.1)
axes[1].legend(
    loc='upper right',
    bbox_to_anchor=(1, 1),
    framealpha=0.7     # 半透明
)

# ACC plot
sns.barplot(x='Model', y='Test Accuracy', data=df_perf, ax=axes[2], palette="Purples")
axes[2].set_title('Test Set Accuracy Comparison')
axes[2].set_ylabel('Accuracy')
axes[2].set_ylim(0.5, 1)
axes[2].tick_params(axis='x', labelsize=16)
axes[2].legend(
    loc='upper right',
    bbox_to_anchor=(1, 1),
    framealpha=0.7     # 半透明
)
for bar in axes[2].containers:
    axes[2].bar_label(bar, fmt='%.4f', label_type='edge', padding=4, fontsize=14)
axes[2].margins(y=0.1)


# 当前工作目录
PROJECT_ROOT = os.getcwd()

if os.path.basename(PROJECT_ROOT) == "02_Programme":
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "03_Output", "ml_anx", "plots_anx_cat")
os.makedirs(OUTPUT_DIR, exist_ok=True)
save_path = os.path.join(OUTPUT_DIR, "Anxiety_Performance_1x3.png")
plt.savefig(save_path, dpi=600, bbox_inches='tight')
plt.close()

print(f"Saved to: {save_path}")

NameError: name 'res_rf' is not defined

In [ ]:
results = {
    'CatBoost': res_cat,
    'LGBM': res_lgbm,
    'RF': res_rf,
    'XGBoost': res_xgb
}

data_list = []

for model_name, res in results.items():
    # Train set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Train',
        'AUC': res['train_auc'],
        'F1': res['train_f1'],
        'Accuracy': res['train_acc']
    })
    
    # Evalidation set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Validation',
        'AUC': res['val_auc'],
        'F1': res['val_f1'],
        'Accuracy': res['val_acc']
    })
    
    # Test set metrics
    data_list.append({
        'Model': model_name,
        'Dataset': 'Test',
        'AUC': res['test_auc'],
        'F1': res['test_f1'],
        'Accuracy': res['test_acc']
    })

# DataFrame
df_final_perf = pd.DataFrame(data_list)
EXCEL_DIR = os.path.join(PROJECT_ROOT, "03_Output", "ml_anx", "output")
os.makedirs(EXCEL_DIR, exist_ok=True)
excel_save_path = os.path.join(EXCEL_DIR, "Single_Model_Full_Performance.xlsx")
df_final_perf = df_final_perf.sort_values(by=['Model', 'Dataset'])

df_final_perf.to_excel(excel_save_path, index=False)

print("\n" + "="*50)
print(f"All model performances had been saved in: {excel_save_path}")
print("="*50)
print("\n=== The comparison of model performance ===")
print(df_final_perf)

# === Plot ===
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'font.size': 18
})

plt.rcParams['font.family'] = 'Times New Roman'

fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=600, tight_layout=True)

# --- 1. AUC ---
sns.barplot(
    x='Model', 
    y='AUC', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[0], 
    palette="Greens"
)
axes[0].set_ylim(0, 1)
axes[0].set_title('AUC Performance Comparison')
axes[0].set_ylabel('AUC Score')
axes[0].tick_params(axis='x')
axes[0].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# --- 2. F1 Score ---
sns.barplot(
    x='Model', 
    y='F1', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[1], 
    palette="Oranges"
)
axes[1].set_ylim(0, 1)
axes[1].set_title('F1 Score Performance Comparison')
axes[1].set_ylabel('F1 Score')
axes[1].tick_params(axis='x')
axes[1].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# --- 3. Accuracy ---
sns.barplot(
    x='Model', 
    y='Accuracy', 
    hue='Dataset', 
    data=df_final_perf, 
    ax=axes[2], 
    palette="Purples"
)
axes[2].set_ylim(0, 1)
axes[2].set_title('Accuracy Performance Comparison')
axes[2].set_ylabel('Accuracy Score')
axes[2].tick_params(axis='x')
axes[2].legend(
        loc = 'upper center',
        bbox_to_anchor=(0.5, 0.9995),
        ncol=3,
        frameon=True,
        fontsize=15.5,
        framealpha=0.7     # 半透明
)

# plt.suptitle(f'Single Model Performance Across Train, Validation, and Test Sets', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

png_save_path = os.path.join(
    OUTPUT_DIR,
    "Performance_Comparison_5Models_Full_Comparison.png"
)

plt.savefig(png_save_path, dpi=600, bbox_inches='tight')
plt.close()

print(f"\nAll performance comparison has been saved in:\n{png_save_path}")


All model performances had been saved in: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\output\Single_Model_Full_Performance.xlsx

=== The comparison of model performance ===
       Model     Dataset       AUC        F1  Accuracy
2   CatBoost        Test  0.822901  0.755043  0.606481
0   CatBoost       Train  0.818129  0.755877  0.607558
1   CatBoost  Validation  0.719328  0.755396  0.606936
5       LGBM        Test  0.831432  0.772455  0.648148
3       LGBM       Train  0.883705  0.801534  0.699128
4       LGBM  Validation  0.713725  0.760456  0.635838
8         RF        Test  0.832869  0.811594  0.759259
6         RF       Train  0.879266  0.835787  0.789244
7         RF  Validation  0.723389  0.780269  0.716763
11   XGBoost        Test  0.821284  0.769874  0.745370
9    XGBoost       Train  0.783267  0.718291  0.693314
10   XGBoost  Validation  0.728992  0.684211  0.653179

All performance comparison has been saved in:
d:\1.SJTU\MentalHealthProjects\ML_CLPN_Proje

In [ ]:
import os
import pandas as pd
import shap
import matplotlib.pyplot as plt
from ML_shap_anx import run_full_shap_analysis

VARIABLE_MAPPING = {
    "somatic_y1": "Somatic Symptoms",
    "BMI_T1_cat": "BMI Category", 
    "sleep_dura_T1_cat": "Sleep Duration",
    "sleep_quali_T1": "Sleep Quality",
    "insomnia_y1": "Insomnia",
    
    "life_satis_y1": "Life Satisfaction",
    "ms_ses_y1": "Mindset of Socioeconomic Status", 
    "per_stress_y1": "Perceived Stress", 
    "ms_stress_y1": "Stress Mindset",
    "depression_y1": "Baseline Depressive Symptoms",
    "anxiety_y1": "BaselineAnxiety Symptoms",
    
    "ace": "Adverse Childhood Experiences",
    
    "loneliness_y1": "Loneliness", 
    "support_y1": "Social Support",
    
    "gender_T1": "Gender",
    "age_T1": "Age",
    "grade_T1": "Grade",
    "residence": "Residence", 
    "income": "Household Income",
    "income_ineqCity_y1": "Perceived Income Inequality",
    "sss_now": "Subjective Socialeconomic Status",
    "marrige_par_bin": "Parental Marital Status",
    "edu_pa": "Parental Educational Level",
    
    "eat_unctl_y1": "Uncontrolled Eating",
    "eat_emot_y1": "Emotional Eating", 
    "food_sweetdrink_T1": "Sugar-Sweetened Beverage Consumption",
    "food_takeout_T1": "Takeaway Frequency",
    
    "IPAQ_T1_1_bin": "Vigorous Physical Activity",
    "IPAQ_T1_3_bin": "Moderate Physical Activity", 
    "IPAQ_T1_5_bin": "Walking Activity",
    
    "screenT_weekday_T1": "Weekday Screen Time",
    "screenT_weekend_T1": "Weekend Screen Time",
    
    "psmu_y1": "Problematic Social Media Use",
    "media_BadMood_T1": "Social Media Posting of Negative Emotion", 
    "media_GoodMood_T1": "Social Media Posting of Positive Emotion",
    
    "edu_self": "Educational Aspiration"
}


all_results = {
    'RandomForest': res_rf,
    'XGBoost': res_xgb,
    'LightGBM': res_lgbm,
    'CatBoost': res_cat
}


FLAG_SHOW = False 
FLAG_TITLE = False 
IS_LOG = False 
TOPN = 15


for name, results in all_results.items():
    run_full_shap_analysis(
        model_name=name,
        results=results,
        df_model=df_model,         
        selected_vars=selected_vars,
        variable_mapping=VARIABLE_MAPPING,
        is_log_transformed=IS_LOG,
        top_n=TOPN,
        flag_show=FLAG_SHOW,
        flag_title=FLAG_TITLE
    )

C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Full SHAP Analysis for RandomForest
-> SHAP values calculated successfully.
Plot saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\plots_anx_cat\shap_summary_simple_randomforest.png
-> 1/6 Plotted Simple Summary.
Plot saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\plots_anx_cat\feature_importance_bar_randomforest.png
-> 2/6 Plotted Feature Importance Bar.
Plot saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\plots_anx_cat\shap_summary_with_bars_randomforest.png
-> 3/6 Plotted Standard SHAP Summary.
Plot saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\plots_anx_cat\shap_dependence_plots_randomforest.png
-> 4/6 Plotted Dependence Plots for 8 features.
-> 5/6 Generated Log-Transformed Interpretation.
Results saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\output\SHAP_Analysis_randomforest_20260328_163045.xlsx
-> 6/6 Saved results to Excel: d:\1.SJTU\Ment

C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\liuxi\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Plot saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\plots_anx_cat\shap_dependence_plots_xgboost.png
-> 4/6 Plotted Dependence Plots for 8 features.
-> 5/6 Generated Log-Transformed Interpretation.
Results saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\output\SHAP_Analysis_xgboost_20260328_163051.xlsx
-> 6/6 Saved results to Excel: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\output\SHAP_Analysis_xgboost_20260328_163051.xlsx
Full SHAP Analysis for XGBoost COMPLETED.
Starting Full SHAP Analysis for LightGBM
-> SHAP values calculated successfully.
Plot saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\plots_anx_cat\shap_summary_simple_lightgbm.png
-> 1/6 Plotted Simple Summary.
Plot saved to: d:\1.SJTU\MentalHealthProjects\ML_CLPN_Project\03_Output\ml_anx\plots_anx_cat\feature_importance_bar_lightgbm.png
-> 2/6 Plotted Feature Importance Bar.
Plot saved to: d:\1.SJTU\MentalHealthProjects\ML_CL